In [14]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [15]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

In [16]:
df_de = df_de.drop(["Unnamed: 0"], axis = 1)

In [17]:
df_de

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,30902.298694,26.46,77.264286,6.710000,6.550000,11.82,10.039,6.389770,17.555365,-0.168868,1.46,-1.2,15.2,0.000010,581.657956,6.196333
1,31539.117534,26.06,77.264286,6.670000,6.540000,10.93,10.072,5.783688,17.951098,-0.295519,0.81,-2.6,15.2,0.000010,599.683352,10.012677
2,30461.535107,23.87,73.200000,6.610000,6.490000,11.31,10.851,4.768184,19.597935,-2.070729,1.63,-2.4,15.5,0.000009,584.358974,37.999363
3,31990.195768,26.06,74.000000,6.610000,6.420000,10.42,11.667,5.103897,19.176892,-3.199506,2.48,-0.6,15.6,0.000010,575.287356,34.793707
4,34229.966763,26.31,75.400000,6.450000,6.150000,8.93,12.470,4.852664,18.693432,-0.370659,2.55,1.3,15.8,0.000010,585.349446,39.857009
5,34819.413660,26.58,75.800000,6.350000,6.090000,8.12,13.549,6.037145,18.897792,0.092987,1.82,2.4,16.1,0.000010,579.634465,40.920931
6,35554.076381,25.91,76.300000,6.080000,6.010000,7.82,13.757,5.772928,19.247998,0.204270,1.76,3.0,16.1,0.000010,587.141971,52.698567
7,36861.914611,26.01,76.700000,5.940000,5.880000,7.61,14.381,6.537450,19.210129,0.684115,0.44,5.3,16.7,0.000010,594.640652,39.295143
8,37774.244301,25.92,77.000000,5.950000,5.720000,7.33,14.901,7.417530,19.312236,0.679371,1.82,12.0,16.7,0.000010,580.046517,27.981817
9,38811.920567,26.22,77.600000,5.900000,5.600000,7.08,14.885,7.263048,19.519072,0.736708,1.64,4.2,16.5,0.000010,588.706740,8.995473


In [18]:
de = pd.DataFrame()
de = df_de.copy()
de['PC_1'] = de['PC'].shift(1)
de['GGFC_2'] = de['GGFC'].shift(2)
de['FDI_2'] = de['FDI'].shift(2)
de['ARP_1'] = de['ARP'].shift(1)
# de['PAM_3'] = de['PAM'].shift(3)
# de['EBSG_3'] = de['EBSG'].shift(3)

In [19]:
de = de.dropna()

In [20]:
sm = StructureModel()

In [21]:
sm.add_edges_from([
    ('EUPC', 'VABI'),
    ('GGFC', 'VABI'),
    ('YUA', 'QOA'),
    ('YUA', 'QOR'),
    ('REST', 'YUA'),
    ('GGFC','REST'),
    ('CO2C', 'GGFC'),
    ('GGFC', 'LBGD'),
    ('EUPC', 'EBSG'),
    ('EUPC', 'PC'),
    ('QOA', 'EUPC'),
    ('PC_1', 'EUPC'),
    ('GGFC_2', 'EUPC'),
    ('FDI_2', 'EUPC'),
    ('ARP_1', 'PC'),
])

In [22]:
sm.edges

OutEdgeView([('EUPC', 'VABI'), ('EUPC', 'EBSG'), ('EUPC', 'PC'), ('GGFC', 'VABI'), ('GGFC', 'REST'), ('GGFC', 'LBGD'), ('YUA', 'QOA'), ('YUA', 'QOR'), ('QOA', 'EUPC'), ('REST', 'YUA'), ('CO2C', 'GGFC'), ('PC_1', 'EUPC'), ('GGFC_2', 'EUPC'), ('FDI_2', 'EUPC'), ('ARP_1', 'PC')])

In [23]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_de_1.html")

Graphs/fully_connected_de_1.html


In [24]:
bn = BayesianNetwork(sm)

In [25]:
discretised_de = pd.DataFrame(index=de.index)

for col in de.columns:
    no_unique = de[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                de[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = de[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_de)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     0    0    2    2    2     0     0     2     0    0   0    0   
3      0     2    0    2    2    2     0     0     0     0    1   0    0   
4      0     2    0    2    2    2     0     0     0     1    2   1    0   
5      0     2    0    2    2    2     0     2     0     1    0   1    1   
6      0     1    0    1    2    1     0     1     0     1    0   2    1   
7      1     1    1    0    1    1     1     2     0     2    0   2    2   
8      1     1    1    1    0    1     1     2     1     2    0   2    2   
9      1     2    1    0    0    1     1     2     1     2    0   2    2   
10     1     2    1    0    0    0     1     2     1     1    2   2    1   
11     2     1    2    0    0    0     2     1     1     2    2   1    1   
12     2     0    2    0    0    0     2     1     2     2    1   1    0   
13     2     0    1    1    1    2     2     1     2     0    2   0   

In [26]:
discretised_de = discretised_de.reset_index(drop=True)
bn.fit_node_states(discretised_de)
baseline_auc = utils.get_avg_auc_all_info(discretised_de, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.354261159896851 seconds
Processing fold 1 using 7 cores takes 5.935548782348633 seconds
Processing fold 2 using 7 cores takes 5.968766927719116 seconds
Processing fold 3 using 7 cores takes 5.6324639320373535 seconds
Processing fold 4 using 7 cores takes 5.542852878570557 seconds
Baseline AUC: 0.5931481481481481


In [27]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'VABI')]
edges_to_remove = [('EUPC', 'VABI')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [28]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_1.html")

Graphs/node_added_de_1.html


In [90]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'VABI': {proposed_auc}")

Processing fold 0 using 7 cores takes 11.346670150756836 seconds
Processing fold 1 using 7 cores takes 10.265918970108032 seconds
Processing fold 2 using 7 cores takes 9.888437032699585 seconds
Processing fold 3 using 7 cores takes 9.78584599494934 seconds
Processing fold 4 using 7 cores takes 10.050530910491943 seconds
AUC from adding LV between 'EUPC' and 'VABI': 0.6322222222222222


In [92]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'EBSG')]
edges_to_remove = [('EUPC', 'EBSG')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [93]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_2.html")

Graphs/node_added_de_2.html


In [94]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'EBSG': {proposed_auc}")

Processing fold 0 using 7 cores takes 8.451637983322144 seconds
Processing fold 1 using 7 cores takes 7.957451820373535 seconds
Processing fold 2 using 7 cores takes 7.716219186782837 seconds
Processing fold 3 using 7 cores takes 7.581306219100952 seconds
Processing fold 4 using 7 cores takes 7.905380010604858 seconds
AUC from adding LV between 'EUPC' and 'EBSG': 0.6565740740740741


In [95]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'PC')]
edges_to_remove = [('EUPC', 'PC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [96]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_3.html")

Graphs/node_added_de_3.html


In [97]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'PC': {proposed_auc}")

Processing fold 0 using 7 cores takes 10.569378137588501 seconds
Processing fold 1 using 7 cores takes 9.877302169799805 seconds
Processing fold 2 using 7 cores takes 10.177395105361938 seconds
Processing fold 3 using 7 cores takes 9.991865158081055 seconds
Processing fold 4 using 7 cores takes 9.996927976608276 seconds
AUC from adding LV between 'EUPC' and 'PC': 0.5729629629629629


In [98]:
edges_to_add = [('LV', 'QOA'), ('LV', 'EUPC')]
edges_to_remove = [('QOA', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [99]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_4.html")

Graphs/node_added_de_4.html


In [100]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'QOA' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 7.5076940059661865 seconds
Processing fold 1 using 7 cores takes 7.0360448360443115 seconds
Processing fold 2 using 7 cores takes 6.852190971374512 seconds
Processing fold 3 using 7 cores takes 7.262394189834595 seconds
Processing fold 4 using 7 cores takes 7.60368800163269 seconds
AUC from adding LV between 'QOA' and 'EUPC': 0.6998148148148148


In [101]:
edges_to_add = [('LV', 'PC_1'), ('LV', 'EUPC')]
edges_to_remove = [('PC_1', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [102]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_5.html")

Graphs/node_added_de_5.html


In [103]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'PC_1' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 7.336175918579102 seconds
Processing fold 1 using 7 cores takes 6.604272842407227 seconds
Processing fold 2 using 7 cores takes 6.468600749969482 seconds
Processing fold 3 using 7 cores takes 6.3835289478302 seconds
Processing fold 4 using 7 cores takes 6.176011085510254 seconds
AUC from adding LV between 'PC_1' and 'EUPC': 0.6016666666666667


In [104]:
edges_to_add = [('LV', 'GGFC_2'), ('LV', 'EUPC')]
edges_to_remove = [('GGFC_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [105]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_6.html")

Graphs/node_added_de_6.html


In [106]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'GGFC_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 7.690584182739258 seconds
Processing fold 1 using 7 cores takes 6.368558645248413 seconds
Processing fold 2 using 7 cores takes 6.6780619621276855 seconds
Processing fold 3 using 7 cores takes 6.536639928817749 seconds
Processing fold 4 using 7 cores takes 6.2727789878845215 seconds
AUC from adding LV between 'GGFC_2' and 'EUPC': 0.6103703703703703


In [107]:
edges_to_add = [('LV', 'FDI_2'), ('LV', 'EUPC')]
edges_to_remove = [('FDI_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [108]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_7.html")

Graphs/node_added_de_7.html


In [109]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'FDI_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 8.240912199020386 seconds
Processing fold 1 using 7 cores takes 6.662366151809692 seconds
Processing fold 2 using 7 cores takes 6.369059801101685 seconds
Processing fold 3 using 7 cores takes 6.5087761878967285 seconds
Processing fold 4 using 7 cores takes 6.633145809173584 seconds
AUC from adding LV between 'FDI_2' and 'EUPC': 0.7009259259259258


In [110]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [111]:
df_de_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

In [112]:
df_de_1 = df_de_1.drop(["Unnamed: 0"], axis = 1)

In [113]:
de_1 = pd.DataFrame()
de_1 = df_de_1.copy()
de_1['EUPC_3'] = de_1['EUPC'].shift(3)
de_1['PAM_3'] = de_1['PAM'].shift(3)
de_1['EBSG_3'] = de_1['EBSG'].shift(3)

In [114]:
de_1 = de_1.dropna()

In [115]:
sm1 = StructureModel()

In [116]:
sm1.add_edges_from([
    ('EUPC_3', 'ARP'),
    ('PAM_3', 'ARP'),
    ('FDI', 'ARP'),
     ('EBSG_3', 'FDI')
])

In [117]:
sm1.edges

OutEdgeView([('EUPC_3', 'ARP'), ('PAM_3', 'ARP'), ('FDI', 'ARP'), ('EBSG_3', 'FDI')])

In [118]:
bn1 = BayesianNetwork(sm1)

In [119]:
discretised_de = pd.DataFrame(index=de_1.index)

for col in de_1.columns:
    no_unique = de_1[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                de_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = de_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_de)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
3      0     1    0    2    2    2     0     0     0     0    1   0    0   
4      0     2    0    2    2    2     0     0     0     0    1   0    0   
5      0     2    0    2    2    2     0     1     0     1    0   1    1   
6      0     0    0    1    2    1     0     1     0     1    0   1    1   
7      0     1    0    0    1    1     0     2     0     2    0   2    2   
8      1     1    1    1    0    1     1     2     1     1    0   2    2   
9      1     2    1    0    0    1     1     2     1     2    0   2    2   
10     1     2    1    0    0    0     1     2     1     1    2   2    1   
11     1     1    2    0    0    0     1     1     1     2    2   1    0   
12     2     0    2    0    0    0     2     1     2     2    1   1    0   
13     2     0    1    1    1    2     2     0     2     0    2   0    1   
14     2     0    2    1    1    0     2     0     2     0    2   0   

In [120]:
discretised_de = discretised_de.reset_index(drop=True)
bn1.fit_node_states(discretised_de)
baseline_auc = utils.get_avg_auc_all_info(discretised_de, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.4610090255737305 seconds
Processing fold 1 using 7 cores takes 5.117887020111084 seconds
Processing fold 2 using 7 cores takes 5.236715793609619 seconds
Processing fold 3 using 7 cores takes 5.260581016540527 seconds
Processing fold 4 using 7 cores takes 5.144036769866943 seconds
Baseline AUC: 0.47972222222222227


In [121]:
edges_to_add = [('LV', 'EUPC_3'), ('LV', 'ARP')]
edges_to_remove = [('EUPC_3', 'ARP')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [122]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_de_5.html")

Graphs/node_added_de_5.html


In [123]:
discretised_de['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_de, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_3' and 'ARP': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.039407968521118 seconds
Processing fold 1 using 7 cores takes 5.464698076248169 seconds
Processing fold 2 using 7 cores takes 5.477513790130615 seconds
Processing fold 3 using 7 cores takes 5.438821792602539 seconds
Processing fold 4 using 7 cores takes 5.485311985015869 seconds
AUC from adding LV between 'EUPC_3' and 'ARP': 0.4852777777777778
